# 🚀 Complete Computer Vision Workflow - Google Colab Ready

## Welcome! 👋

This notebook provides a **complete end-to-end** Computer Vision solution that runs entirely in Google Colab!

### What You'll Do:
1. 🎓 Train a CNN model on CIFAR-10 dataset
2. 📹 Set up real-time webcam classification
3. 🌐 Deploy a web app using Flask + ngrok
4. 🎉 Access your classifier from any device!

### Prerequisites:
- ✅ Google account (to use Colab)
- ✅ Webcam on your device
- ✅ 10-15 minutes of time

### 💡 Pro Tips:
- Run cells in order (don't skip!)
- Use GPU runtime for faster training (Runtime → Change runtime type → GPU)
- Read the comments - they explain what each step does!

Let's get started! 🎯

---
## Part 1: Setup and Installation

First, let's install all required packages.

In [ ]:
%%capture
# Install required packages
!pip install flask-ngrok pyngrok flask opencv-python-headless tensorflow

In [ ]:
# Import all required libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from IPython.display import display, HTML, Image as IPImage
import base64
import io
from google.colab.output import eval_js
from google.colab import output

print("✅ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Part 2: Load and Explore CIFAR-10 Dataset

CIFAR-10 contains 60,000 images in 10 classes.

In [ ]:
# Load CIFAR-10 dataset
print("📦 Loading CIFAR-10 dataset...")
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Define class names
CLASS_NAMES = [
    'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
    'Dog', 'Frog', 'Horse', 'Ship', 'Truck'
]

print(f"\n✅ Dataset loaded successfully!")
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")
print(f"Image shape: {X_train.shape[1:]}")
print(f"Number of classes: {len(CLASS_NAMES)}")

In [ ]:
# Visualize sample images
plt.figure(figsize=(12, 6))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i])
    plt.xlabel(CLASS_NAMES[y_train[i][0]], fontsize=9)
plt.suptitle('Sample CIFAR-10 Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3: Preprocess Data

Normalize pixel values to improve training.

In [ ]:
# Normalize pixel values to [0, 1]
X_train_normalized = X_train.astype('float32') / 255.0
X_test_normalized = X_test.astype('float32') / 255.0

print(f"✅ Data normalized")
print(f"Original range: [{X_train.min()}, {X_train.max()}]")
print(f"Normalized range: [{X_train_normalized.min()}, {X_train_normalized.max()}]")

---
## Part 4: Build CNN Model

Create a Convolutional Neural Network for image classification.

In [ ]:
def create_cnn_model():
    """
    Build a CNN model for CIFAR-10 classification.
    
    Architecture:
    - 3 Convolutional blocks (Conv2D + MaxPooling + Dropout)
    - 2 Dense layers
    - Dropout for regularization
    """
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                     input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Output layer
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Create the model
model = create_cnn_model()

# Display model architecture
print("🏗️ Model Architecture:")
print("="*60)
model.summary()
print("="*60)

---
## Part 5: Compile and Train Model

This will take ~5-10 minutes depending on your runtime.

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled successfully!")

In [ ]:
# Train the model
print("🏋️ Training started...\n")

history = model.fit(
    X_train_normalized, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_normalized, y_test),
    verbose=1
)

print("\n✅ Training completed!")

---
## Part 6: Evaluate Model Performance

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test_normalized, y_test, verbose=0)

print("\n" + "="*50)
print("📊 MODEL PERFORMANCE")
print("="*50)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print("="*50)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Training', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Training', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 7: Save the Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save the model
model.save('models/cifar10_model.h5')
print("✅ Model saved to 'models/cifar10_model.h5'")

# Save in SavedModel format too
model.save('models/cifar10_saved_model')
print("✅ Model also saved in SavedModel format")

---
## Part 8: Test Model with Sample Predictions

In [ ]:
# Make predictions on test images
predictions = model.predict(X_test_normalized[:16])

# Visualize predictions
plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_test[i])
    
    predicted_label = np.argmax(predictions[i])
    true_label = y_test[i][0]
    confidence = predictions[i][predicted_label] * 100
    
    color = 'green' if predicted_label == true_label else 'red'
    
    plt.xlabel(f"{CLASS_NAMES[predicted_label]}\n{confidence:.1f}%",
              color=color, fontsize=9)

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', 
            fontsize=14, y=1.0, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 9: Setup Webcam Classification Functions

Now let's create functions to classify images from your webcam!

In [ ]:
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL
import io

def take_photo(filename='photo.jpg', quality=0.8):
    """
    Take a photo using the webcam in Google Colab.
    
    Args:
        filename: Name to save the photo
        quality: Image quality (0-1)
    
    Returns:
        PIL Image
    """
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = '📸 Capture Photo';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    
    # Get the image data
    data = eval_js('takePhoto({})'.format(quality))
    
    # Decode and save
    binary = b64decode(data.split(',')[1])
    
    with open(filename, 'wb') as f:
        f.write(binary)
    
    return PIL.Image.open(io.BytesIO(binary))

def classify_image(img):
    """
    Classify an image using the trained model.
    
    Args:
        img: PIL Image or numpy array
    
    Returns:
        tuple: (predicted_class, confidence, all_predictions)
    """
    # Convert PIL to numpy if needed
    if isinstance(img, PIL.Image.Image):
        img = np.array(img)
    
    # Resize to 32x32
    img_resized = cv2.resize(img, (32, 32))
    
    # Convert to RGB if needed
    if len(img_resized.shape) == 2:
        img_resized = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
    elif img_resized.shape[2] == 4:
        img_resized = cv2.cvtColor(img_resized, cv2.COLOR_RGBA2RGB)
    
    # Normalize
    img_normalized = img_resized.astype('float32') / 255.0
    
    # Add batch dimension
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    # Predict
    predictions = model.predict(img_batch, verbose=0)
    
    predicted_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_idx]
    
    return CLASS_NAMES[predicted_idx], confidence, predictions[0]

print("✅ Webcam functions ready!")

---
## Part 10: Capture and Classify!

🎥 **Click the button below to take a photo and classify it!**

In [ ]:
# Take photo from webcam
print("📸 Click 'Capture Photo' button to take a picture...\n")
img = take_photo()

# Classify the image
predicted_class, confidence, all_preds = classify_image(img)

# Display results
print("\n" + "="*50)
print("🎯 CLASSIFICATION RESULT")
print("="*50)
print(f"Predicted: {predicted_class}")
print(f"Confidence: {confidence*100:.2f}%")
print("="*50)

# Show image
plt.figure(figsize=(10, 5))

# Original image
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f"Captured Image", fontsize=12, fontweight='bold')
plt.axis('off')

# Prediction probabilities
plt.subplot(1, 2, 2)
y_pos = np.arange(len(CLASS_NAMES))
plt.barh(y_pos, all_preds * 100)
plt.yticks(y_pos, CLASS_NAMES)
plt.xlabel('Confidence (%)', fontsize=11)
plt.title('Prediction Probabilities', fontsize=12, fontweight='bold')
plt.xlim(0, 100)

# Highlight the top prediction
max_idx = np.argmax(all_preds)
plt.gca().get_children()[max_idx].set_color('green')

plt.tight_layout()
plt.show()

print("\n💡 Tip: Run this cell again to classify another image!")

---
## Part 11: Deploy Flask App with ngrok

Now let's create a real-time web app accessible from anywhere!

In [ ]:
# Install ngrok
!pip install pyngrok -q

from pyngrok import ngrok
from flask import Flask, render_template_string, Response, jsonify
import threading

print("✅ ngrok installed successfully!")

In [ ]:
# Create Flask app
app = Flask(__name__)

# HTML template for the web interface
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html>
<head>
    <title>Real-Time Image Classifier</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            text-align: center;
            padding: 20px;
        }
        .container {
            max-width: 800px;
            margin: 0 auto;
            background: white;
            color: #333;
            border-radius: 20px;
            padding: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.3);
        }
        h1 { color: #667eea; margin-bottom: 10px; }
        .video-container {
            margin: 20px 0;
            border-radius: 15px;
            overflow: hidden;
            box-shadow: 0 5px 15px rgba(0,0,0,0.2);
        }
        #webcam { width: 100%; max-width: 640px; }
        #result {
            font-size: 24px;
            margin: 20px 0;
            padding: 15px;
            background: #f0f0f0;
            border-radius: 10px;
            font-weight: bold;
        }
        .confidence { color: #667eea; }
        .info {
            background: #e8f4f8;
            padding: 15px;
            border-radius: 10px;
            margin-top: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🎥 Real-Time Image Classifier</h1>
        <p>Powered by CIFAR-10 CNN Model</p>
        
        <div class="video-container">
            <video id="webcam" autoplay playsinline></video>
        </div>
        
        <canvas id="canvas" style="display:none;"></canvas>
        
        <div id="result">📸 Ready to classify!</div>
        
        <div class="info">
            <p><strong>🏷️ Can recognize:</strong> Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck</p>
        </div>
    </div>
    
    <script>
        const video = document.getElementById('webcam');
        const canvas = document.getElementById('canvas');
        const result = document.getElementById('result');
        const ctx = canvas.getContext('2d');
        
        // Access webcam
        navigator.mediaDevices.getUserMedia({ video: true })
            .then(stream => {
                video.srcObject = stream;
                // Start classification
                setInterval(classify, 1000);
            })
            .catch(err => {
                result.innerHTML = "❌ Error: Cannot access webcam";
            });
        
        async function classify() {
            // Capture frame
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            ctx.drawImage(video, 0, 0);
            
            // Convert to blob
            canvas.toBlob(async (blob) => {
                const formData = new FormData();
                formData.append('image', blob);
                
                try {
                    const response = await fetch('/predict', {
                        method: 'POST',
                        body: formData
                    });
                    const data = await response.json();
                    
                    result.innerHTML = `🎯 <strong>${data.class}</strong><br>
                        <span class="confidence">Confidence: ${data.confidence}%</span>`;
                } catch (err) {
                    console.error('Classification error:', err);
                }
            }, 'image/jpeg');
        }
    </script>
</body>
</html>
'''

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    from flask import request
    
    # Get image from request
    file = request.files['image']
    img = PIL.Image.open(file.stream)
    
    # Classify
    predicted_class, confidence, _ = classify_image(img)
    
    return jsonify({
        'class': predicted_class,
        'confidence': f"{confidence*100:.2f}"
    })

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

print("✅ Flask app created!")

In [ ]:
# Set up ngrok tunnel
public_url = ngrok.connect(5000)

print("\n" + "="*60)
print("🌐 WEB APP IS LIVE!")
print("="*60)
print(f"\n🔗 Access your app at: {public_url}")
print("\n💡 Copy the link above and open it in your browser!")
print("📱 You can access it from any device (phone, tablet, etc.)")
print("\n⚠️  Note: The link will stop working when you stop this notebook")
print("="*60)

# Display clickable link
display(HTML(f'''
<div style="background: #4CAF50; padding: 20px; border-radius: 10px; text-align: center;">
    <h2 style="color: white; margin: 0;">🎉 Your App is Ready!</h2>
    <p style="color: white; margin: 10px 0;">
        <a href="{public_url}" target="_blank" 
           style="color: #FFD700; font-size: 20px; text-decoration: none; font-weight: bold;">
            👉 CLICK HERE TO OPEN YOUR APP 👈
        </a>
    </p>
</div>
'''))

In [ ]:
# Run Flask app
# Note: This will keep running until you stop it
print("\n🚀 Starting server...\n")
print("⚠️  To stop the server: Runtime → Interrupt execution\n")

app.run(port=5000)

---
## 🎉 Congratulations!

You've successfully:
- ✅ Trained a CNN model on CIFAR-10
- ✅ Created a real-time webcam classifier
- ✅ Deployed a web app accessible from anywhere

## 🚀 Next Steps

1. **Improve the Model:**
   - Train for more epochs
   - Add data augmentation
   - Try different architectures

2. **Try Other Datasets:**
   - MNIST (handwritten digits)
   - Fashion-MNIST (clothing)
   - Your own custom dataset!

3. **Advanced Features:**
   - Object detection with YOLO
   - Face recognition
   - Image segmentation

## 📚 Learning Resources

- [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
- [Keras Documentation](https://keras.io/)
- [Deep Learning Specialization](https://www.coursera.org/specializations/deep-learning)

## 🤝 Share Your Work!

Share your app link with friends and family to show them what you built!

---

**Happy Learning! 🎓**

*Built with ❤️ for Computer Vision beginners*